#**Trabalho final - Classificador de produtos com GenAI**

###**Caso de uso:**

Uma empresa de marketplace, disponibiliza sua plataforma para diversos vendedores cadastrarem seus produtos em diferentes categorias previamente definidas. Essas categorias são utilizadas para melhor distribuir e divulgar seus produtos para os clientes e usuários da plataforma. Mas nem todos os vendedores respeitam essas categorias, regras e as diretrizes do marketplace.

Pense nos diversos problemas que podemos enfrentar:
- Vendedores que cadastram produtos em categorias erradas;
- Vendedores que querem vender produtos ilícitos;
- Vendedores que querem vender produtos que não são permitidos pelas políticas do marketplace e por aí vai...

**Será que é possível validar o cadastro desses produtos e categorias, produto por produto**? Um por um?... Que trampo, não???

###**Desafio:**
- Conseguimos ajudar a mitigar parte desse problema?
- Você conseguiria criar algum **processo** que diminua esse trabalho manual e que **valide a categoria** desses produtos?

Bom, podemos criar um **processo que seja capaz de classificar um produto através do nome e da descrição**, e depois podemos confrontar com a categoria e premissas da plataforma usando **IA Generativa e modelos LLMs**.

###**Orientações:**

---
####**Usem o Google Colab com Python e esse template para desenvolverem o trabalho.**
---

**1. Análise exploratória**, vamos começar explorando a base de dados de produtos [1]?
  - Faça uma análise exploratória para entender os dados e estrutura do dataframe.
  - Crie uma nova coluna no dataframe chamada **`texto`** concatenando as colunas **`nome`** e **`descricao`**. Ex.: **`nome + " " + descricao`**.
  - Crie um dataframe novo e selecionando uma amostra aleatória de **100 registros** usando o `random_state = 42` (veja o exemplo abaixo).  

**Obs**.:
  - É importante manter essa configuração da amostra para os trabalhos ficarem comparáveis e classificando os mesmos produtos.
  - Faça a análise exploratória com a base completa e não apenas com o sample.

**2. Desenvolvimento e testes**, nessa parte é onde vocês podem explorar o desenvolvimento do trabalho aplicando as técnicas, ferramentas e serviços de GenAI.  
  - Explorem diferentes formas de tratar o problema. Comece selecionando casos individuais para testar as ferramentas, framework, APIs e técnicas de prompt engineering.
  - Fiquem à vontade para explorar os serviços e frameworks vistos em aula ou outros: API da OpenAI Platform, API do Groq, API da Azure AI Foundry, LangChain e outros.
  - Sejam criativos, mas não precisam de muita complexidade e podem explorar outras formas de desenvolver.
  - Explique as decisões e racional do desenvolvimento. Abuse dos comentários.

**3. Processo final**, aqui nessa parte separe apenas o processo final com um pipeline completo para classificar a categoria dos produtos.
  - Criem uma função que receba o `texto` (nome + descrição do produto) e retorne a categoria de classificação do texto (produto).
  - **Apliquem essa função apenas no dataframe com os 100 casos selecionados**.
  - Resultado esperado: No dataframe de produto com os 100 casos, crie uma nova coluna de categoria: `categoria_genai`.
  - Validem se a classificação da Ia Generativa de cada produto está correta ou divergente comparando com a coluna `categoria` escolhida pelo vendedor. Crie uma nova coluna `validacao_categoria` com os valores da validação: `correta` ou `divergente`.
  - **Gerem um arquivo .csv com os 100 casos avaliados com a seguinte estrutura: `nome;descricao;categoria;texto;categoria_genai;validacao_categoria`.**

###**[1] Base de dados de produtos - https://dados-ml-pln.s3-sa-east-1.amazonaws.com/produtos.csv**


PREFIRAM UTILIZAR AS PRÓPRIAS CREDENCIAIS.

**Não divulguem ou publiquem publicamente suas credenciais.**


###**API Key da OpenAI**

In [34]:
OPENAI_API_KEY = "" # @param {"type":"string"}

###**API Key da OpenAI via Azure AI Foundry**

Exemplo de como usar o serviço da OpenAI via Azure OpenAI.

- [Documentação Azure AI Foundry](https://learn.microsoft.com/pt-br/azure/ai-foundry/what-is-azure-ai-foundry)
- [Repositório da OpenAI](https://github.com/openai/openai-python?tab=readme-ov-file#microsoft-azure-openai)


In [35]:
AZURE_OPENAI_API_KEY = "" # @param {"type":"string"}
AZURE_ENDPOINT = "" # @param {"type":"string"}
AZURE_API_VERSION = "2025-01-01-preview" # @param {"type":"string"}
AZURE_DEPLOYMENT_MODEL = "gpt-4.1"# @param {"type":"string"}

```python
# EXEMPLO DE USO DA AZURE COM OPENAI
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key = AZURE_OPENAI_API_KEY,
    api_version = AZURE_API_VERSION,
    azure_endpoint = AZURE_ENDPOINT,
)

completion = client.chat.completions.create(
    model = AZURE_DEPLOYMENT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Quem é a OpenAI?",
        },
    ],
)

print(completion.to_json())
print(completion.choices[0].message.content)
```

---
#**EXECUÇÃO**

##**1. Análise exploratória**

###**1.1. Objetivo**

Entender a estrutura da base e preparar os dados para a classificação e validação automatizada com IA.

###**1.2. Carregamento da base**
Verifica o tamanho da base e visualiza os primeiros registros para entender o conteúdo.

In [3]:
# ============================================
# Extração - Base de Dados de Produtos (Completa)
# ============================================

import pandas as pd
import numpy as np

# Load da base completa de produtos a partir da fonte externa (CSV)
df = pd.read_csv(
    "https://dados-ml-pln.s3-sa-east-1.amazonaws.com/produtos.csv",
    delimiter=";",
    encoding='utf-8')

In [4]:
# ============================================
# Análise - Dimensão e visualização da base
# ============================================

print("Dimensão da base:")
print(f"\tTotal de Linhas: {df.shape[0]}")
print(f"\tTotal de Colunas: {df.shape[1]}")
print("\nVisualização dos primeiros 10 registros:")
display(df.head(10))


Dimensão da base:
	Total de Linhas: 4080
	Total de Colunas: 3

Visualização dos primeiros 10 registros:


,nome,descricao,categoria
0,O Hobbit - 7ª Ed. 2013,Produto NovoBilbo Bolseiro é um hobbit que lev...,livro
1,Livro - It A Coisa - Stephen King,Produto NovoDurante as férias escolares de 195...,livro
2,Box As Crônicas De Gelo E Fogo Pocket 5 Li...,Produto NovoTodo o reino de Westeros ao alcanc...,livro
3,Box Harry Potter,Produto Novo e Físico A série Harry Potter ch...,livro
4,Livro Origem - Dan Brown,Produto NovoDe Onde Viemos? Para Onde Vamos? R...,livro
5,Mais Escuro - Cinquenta Tons Mais Escuros Pel...,Produto Novo e Físico O relacionamento quente...,livro
6,O Silmarillion - 5ª Ed. 2011,"Produto NovoO Silmarillion, relata acontecimen...",livro
7,O Pequeno Principe,O Pequeno Príncipe é um dos personagens mais f...,livro
8,Ed & Lorraine Warren - Demonologistas Arquiv...,Produto NovoEles enfrentaram os mistérios mais...,livro
9,Box - Franz Kafka 1883-1924 - 3 Livros,Produto NovoEste box contém 3 livros de Franz ...,livro


### **1.3. Estrutura e qualidade dos dados**
Avalia tipos de dados e identifica possíveis valores nulos que possam impactar a análise.

In [5]:
# ============================================
# Análise - Estrutura e qualidade dos dados
# ============================================

df.info()

print("\nValores nulos:")
df_nulos = df.isnull().sum().to_frame(name="qtd_nulos")
print(df_nulos)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4080 entries, 0 to 4079
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   nome       4080 non-null   object
 1   descricao  2916 non-null   object
 2   categoria  4080 non-null   object
dtypes: object(3)
memory usage: 95.8+ KB

Valores nulos:
           qtd_nulos
nome               0
descricao       1164
categoria          0


### **1.4. Dados duplicados**
Verifica se existem registros duplicados que possam distorcer as análises e os resultados.

In [6]:
# ============================================
# Análise - Dados duplicados
# ============================================

qtd_duplicados = df.duplicated().sum()
print(f"Quantidade de registros duplicados: {qtd_duplicados}")

Quantidade de registros duplicados: 291


### **1.5. Categorias existentes**
Analisa quantas categorias existem e como os produtos estão distribuídos entre elas e identifica possíveis desbalanceamentos que podem influenciar o desempenho do modelo.


In [7]:
# ============================================
# Análise - Categorias existentes
# ============================================

# Quantidade de categorias únicas
print(f"Quantidade de categorias únicas: {df['categoria'].nunique()}")

# Distribuição
df_qtd_categoria = df["categoria"].value_counts().to_frame("quantidade").reset_index()
df_qtd_categoria.columns = ["categoria", "quantidade"]

df_qtd_categoria["percentual"] = (
    df_qtd_categoria["quantidade"] / len(df) * 100
).round(2)

print(f"\n Distribuição:")
display(df_qtd_categoria)

Quantidade de categorias únicas: 4

 Distribuição:


,categoria,quantidade,percentual
0,livro,1020,25.0
1,brinquedo,1020,25.0
2,maquiagem,1020,25.0
3,game,1020,25.0


### **1.6. Criação da variável de entrada (texto)**
Adiciona uma nova coluna (texto) concatenando nome e descrição do produto, para fornecer mais contexto ao modelo de IA.

In [8]:
# ============================================
# Criação da variável de entrada (texto)
# ============================================

df["texto"] = df["nome"].fillna("") + " " + df["descricao"].fillna("")
df["texto"] = df["texto"].str.strip()

display(df.head(10))

,nome,descricao,categoria,texto
0,O Hobbit - 7ª Ed. 2013,Produto NovoBilbo Bolseiro é um hobbit que lev...,livro,O Hobbit - 7ª Ed. 2013 Produto NovoBilbo Bols...
1,Livro - It A Coisa - Stephen King,Produto NovoDurante as férias escolares de 195...,livro,Livro - It A Coisa - Stephen King Produto Nov...
2,Box As Crônicas De Gelo E Fogo Pocket 5 Li...,Produto NovoTodo o reino de Westeros ao alcanc...,livro,Box As Crônicas De Gelo E Fogo Pocket 5 Liv...
3,Box Harry Potter,Produto Novo e Físico A série Harry Potter ch...,livro,Box Harry Potter Produto Novo e Físico A sér...
4,Livro Origem - Dan Brown,Produto NovoDe Onde Viemos? Para Onde Vamos? R...,livro,Livro Origem - Dan Brown Produto NovoDe Onde ...
5,Mais Escuro - Cinquenta Tons Mais Escuros Pel...,Produto Novo e Físico O relacionamento quente...,livro,Mais Escuro - Cinquenta Tons Mais Escuros Pelo...
6,O Silmarillion - 5ª Ed. 2011,"Produto NovoO Silmarillion, relata acontecimen...",livro,O Silmarillion - 5ª Ed. 2011 Produto NovoO Si...
7,O Pequeno Principe,O Pequeno Príncipe é um dos personagens mais f...,livro,O Pequeno Principe O Pequeno Príncipe é um do...
8,Ed & Lorraine Warren - Demonologistas Arquiv...,Produto NovoEles enfrentaram os mistérios mais...,livro,Ed & Lorraine Warren - Demonologistas Arquivo...
9,Box - Franz Kafka 1883-1924 - 3 Livros,Produto NovoEste box contém 3 livros de Franz ...,livro,Box - Franz Kafka 1883-1924 - 3 Livros Produt...


### **1.7. Análise do tamanho dos textos**
Verifica o tamanho dos textos, pois descrições curtas podem dificultar a classificação

In [9]:
# ============================================
# Análise - Tamanho dos textos
# ============================================

tamanhos = df["texto"].apply(len)

print("Média:", tamanhos.mean())
print("Mínimo:", tamanhos.min())
print("Máximo:", tamanhos.max())

Média: 890.7904411764706
Mínimo: 12
Máximo: 40670


### **1.8. Criação da amostra (100 registros)**
Seleciona uma amostra fixa para garantir comparabilidade dos testes

In [10]:
# ============================================
# Análise - Criação da amostra (100 registros)
# ============================================

df_sample = df.sample(100, random_state=42)

display(df_sample.head(10))


,nome,descricao,categoria,texto
33,Extraordinário,Produto Novo“Extraordinário” é um livro que co...,livro,Extraordinário Produto Novo“Extraordinário” é...
3316,Fifa 2018 Narração Português Completo Midia D...,FIFA 2018 - XBOX 360 MIDIA DIGITAL COMPLETOATE...,game,Fifa 2018 Narração Português Completo Midia Di...
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,Descrição do Kit Ninja (Foto):Uma Kunai Asuma ...,brinquedo,Kit Mega Sarutobi Kunai Naruto Asuma Shurikens...
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,CAIXA C/ 50 CORES DIFERENTES!BATOM MATTE QUEEN...,maquiagem,Caixa 50 Cores Batom Queen Fosco Matte Nude Ki...
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,A série Crônicas de Gelo e apresentações. São ...,livro,Box Dourado Crónica De Gelo E Fogo Edição De C...
1444,Dragon Ball Z Majin Boo - Shodo Bandai,- Dragon Ball Z Shodo- Figura Majin Boo- Fabri...,brinquedo,Dragon Ball Z Majin Boo - Shodo Bandai - Drag...
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,O ENVIO MAIS RÁPIDO DO MERCADO LIVRE!DE 1 A 12...,brinquedo,Bonecos Vingadores Marvel Avengers Dc Luz E So...
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,POR FAVOR NÃO ESQUEÇA DE SELECIONAR A OPÇÃO RE...,game,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo D...
4005,Pes 2018 Original Ps3 100% Dublado Português ...,DescriçãoAqui Nascem Lendas' apresenta a volta...,game,Pes 2018 Original Ps3 100% Dublado Português B...
1736,Base- Stand -display Para Action Figure-pront...,Base -stand-Display para action figure -figura...,brinquedo,Base- Stand -display Para Action Figure-pronta...


A análise exploratória mostra que a base possui boa estrutura para os experimentos, mas também apresenta limitações típicas de ambientes reais, como descrições incompletas e possíveis inconsistências de categorização. Isso reforça a necessidade de um processo automatizado de validação.

##**2. Desenvolvimento e testes**

### **2.1. Objetivo**

Testar o uso de IA Generativa para classificar produtos a partir do nome e da descrição e avaliar qual abordagem melhor apoia a validação dos cadastros, ajudando a identificar possíveis inconsistências entre o texto do produto e a categoria informada pelo vendedor.

### **2.2. Setup e validação do ambiente**
O ambiente foi configurado para uso do Azure OpenAI e do framework LangChain. A escolha dessas ferramentas foi motivada pela necessidade de acessar modelos de linguagem de forma estruturada, além de facilitar a construção e organização dos prompts utilizados ao longo dos testes.

A validação foi feita em duas etapas:
*   teste de conexão com o Azure OpenAI
*   teste de uso do modelo via LangChain

Essa separação é importante porque cada etapa valida uma parte diferente do fluxo:
*   a conexão garante que o acesso ao serviço está funcionando (credenciais, endpoint, rede)
*   o uso via LangChain garante que a integração com o framework está correta e que os prompts serão executados como esperado

Dessa forma, evitamos erros durante os experimentos e garantimos que qualquer problema posterior esteja relacionado ao modelo ou ao prompt, e não à configuração do ambiente.

In [11]:
# ============================================
# Setup do ambiente - Instalação de dependências (condicional)
# ============================================
import importlib

def instalar_se_nao_existir(pacote, nome_import=None):
    nome_import = nome_import or pacote
    if importlib.util.find_spec(nome_import) is None:
        print(f"Instalando {pacote}...")
        !pip install -q {pacote}

instalar_se_nao_existir("openai")
instalar_se_nao_existir("langchain")
instalar_se_nao_existir("langchain-openai", "langchain_openai")

In [12]:
# ============================================
# Validação do ambiente - Teste de conexão com Azure OpenAI
# ============================================

from openai import AzureOpenAI

client = AzureOpenAI(
    api_key = AZURE_OPENAI_API_KEY,
    api_version = AZURE_API_VERSION,
    azure_endpoint = AZURE_ENDPOINT,
)

completion = client.chat.completions.create(
    model = AZURE_DEPLOYMENT_MODEL,
    messages=[
        {"role": "user", "content": "Responda em uma frase: o que é a OpenAI?"}
    ],
)

print(completion.choices[0].message.content)

A OpenAI é uma empresa de pesquisa em inteligência artificial que desenvolve tecnologias avançadas de IA, como o ChatGPT.


In [13]:
# ============================================
# Validação do ambiente - Teste de uso com LangChain
# ============================================
## O modelo tem temperatura 0 para reduzir variações e tornar as respostas mais consistentes

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_ENDPOINT,
    api_version=AZURE_API_VERSION,
    azure_deployment=AZURE_DEPLOYMENT_MODEL,
    temperature=0)

prompt_teste = ChatPromptTemplate.from_template(
    "Responda de forma objetiva: {pergunta}")

chain_teste = prompt_teste | llm

response = chain_teste.invoke({
    "pergunta": "Quem é a OpenAI?"})

print(response.content)

A OpenAI é uma empresa de pesquisa e desenvolvimento em inteligência artificial fundada em 2015, nos Estados Unidos, com o objetivo de criar e promover IA de forma segura e benéfica para a humanidade.


### **2.3. Definição da abordagem de classificação**

O objetivo é criar um processo automatizado capaz de classificar produtos a partir do nome e da descrição e, em seguida, apoiar a validação da categoria atribuída pelo vendedor.

Esse processo funciona como um mecanismo de apoio à revisão de cadastros, reduzindo o esforço manual e ajudando a identificar possíveis inconsistências.

Para isso, será utilizada uma abordagem baseada em prompting com LLM, onde o modelo recebe o texto do produto e retorna a categoria mais adequada dentro de um conjunto pré-definido.

Serão testadas duas estratégias:
- **Prompt simples**: instrução direta e objetiva
- **Prompt refinado**: instrução mais detalhada, com regras adicionais e maior rigor na decisão

O objetivo da comparação é avaliar qual abordagem melhor contribui para o problema do marketplace: não apenas classificar, mas também sinalizar casos potencialmente inconsistentes.

### **2.4. Preparação da amostra para testes**

Seleciona uma amostra reprodutível e cria uma cópia para trabalhar sem alterar o dataframe original

In [14]:
# ============================================
# Preparação da amostra para testes
# ============================================

df_sample = df.sample(100, random_state=42).copy()
display(df_sample.head(10))

,nome,descricao,categoria,texto
33,Extraordinário,Produto Novo“Extraordinário” é um livro que co...,livro,Extraordinário Produto Novo“Extraordinário” é...
3316,Fifa 2018 Narração Português Completo Midia D...,FIFA 2018 - XBOX 360 MIDIA DIGITAL COMPLETOATE...,game,Fifa 2018 Narração Português Completo Midia Di...
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,Descrição do Kit Ninja (Foto):Uma Kunai Asuma ...,brinquedo,Kit Mega Sarutobi Kunai Naruto Asuma Shurikens...
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,CAIXA C/ 50 CORES DIFERENTES!BATOM MATTE QUEEN...,maquiagem,Caixa 50 Cores Batom Queen Fosco Matte Nude Ki...
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,A série Crônicas de Gelo e apresentações. São ...,livro,Box Dourado Crónica De Gelo E Fogo Edição De C...
1444,Dragon Ball Z Majin Boo - Shodo Bandai,- Dragon Ball Z Shodo- Figura Majin Boo- Fabri...,brinquedo,Dragon Ball Z Majin Boo - Shodo Bandai - Drag...
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,O ENVIO MAIS RÁPIDO DO MERCADO LIVRE!DE 1 A 12...,brinquedo,Bonecos Vingadores Marvel Avengers Dc Luz E So...
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,POR FAVOR NÃO ESQUEÇA DE SELECIONAR A OPÇÃO RE...,game,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo D...
4005,Pes 2018 Original Ps3 100% Dublado Português ...,DescriçãoAqui Nascem Lendas' apresenta a volta...,game,Pes 2018 Original Ps3 100% Dublado Português B...
1736,Base- Stand -display Para Action Figure-pront...,Base -stand-Display para action figure -figura...,brinquedo,Base- Stand -display Para Action Figure-pronta...


### **2.5. Preparação para os testes**
Define as categorias válidas e funções auxiliares para padronizar e validar as respostas do modelo, garantindo consistência na comparação dos resultados.

In [15]:
# ============================================
# Preparação para os testes - Categorias válidas
# ============================================

categorias_validas = sorted(df_sample["categoria"].dropna().unique().tolist())
print(categorias_validas)

['brinquedo', 'game', 'livro', 'maquiagem']


In [16]:
# ============================================
# Funções auxiliares
# ============================================

def normalizar_categoria(texto):
    # Padroniza a saída do modelo (lowercase, remoção de caracteres) para evitar erros na comparação
    if texto is None:
        return ""
    return (
        str(texto)
        .strip()
        .lower()
        .replace(".", "")
        .replace("`", "")
        .replace('"', "")
    )

def validar_resposta_categoria(resposta, categorias_validas):
    # Verifica se a resposta do modelo está entre as categorias válidas.
    # Caso contrário, retorna "fora_escopo"
    resposta = normalizar_categoria(resposta)
    if resposta in categorias_validas:
        return resposta
    return "fora_escopo"

### **2.6. Teste 1 - Prompt simples**
O primeiro teste utiliza um prompt direto, com instruções mínimas, para avaliar o desempenho básico do modelo e estabelecer uma linha de base para comparação com abordagens mais elaboradas.

In [17]:
# ============================================
# Teste 1 - Prompt simples
# ============================================

prompt_simples = ChatPromptTemplate.from_template("""
Você é um classificador de produtos.

Classifique o produto em uma das categorias abaixo:
{categorias}

Regras:
- Responda somente com uma categoria.
- Não explique.
- Não crie novas categorias.
- Se não se encaixar, responda: fora_escopo.

Produto:
{texto}
""")

# Conecta o prompt ao modelo para execução
chain_simples = prompt_simples | llm

# Executa o prompt e valida a resposta do modelo
def classificar_produto_prompt_simples(texto):
    response = chain_simples.invoke({
        "categorias": ", ".join(categorias_validas),
        "texto": texto
    })
    return validar_resposta_categoria(response.content, categorias_validas)

# Teste: comportamento do modelo
df_teste_simples = df_sample.head(10).copy()
df_teste_simples["categoria_ia_simples"] = df_teste_simples["texto"].apply(classificar_produto_prompt_simples)

print("Teste - comparação (10 casos):")
display(df_teste_simples[["nome", "categoria", "categoria_ia_simples"]])

# Aplicação na amostra completa
df_sample["categoria_ia_simples"] = df_sample["texto"].apply(classificar_produto_prompt_simples)

print("\nAmostra completa com classificação:")
display(df_sample[["nome", "categoria", "categoria_ia_simples"]].head(10))

# Comparação da categoria original vs. IA
df_sample["match_simples"] = df_sample["categoria"] == df_sample["categoria_ia_simples"]

print("\nComparação (categoria original vs IA):")
display(df_sample[["nome", "categoria", "categoria_ia_simples", "match_simples"]].head(10))

# Acurácia: avaliação realizada na amostra de 100 registros
acuracia_simples = df_sample["match_simples"].mean()
print(f"\nAcurácia do modelo (prompt simples): {acuracia_simples:.2%}")

Teste - comparação (10 casos):


,nome,categoria,categoria_ia_simples
33,Extraordinário,livro,livro
3316,Fifa 2018 Narração Português Completo Midia D...,game,game
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo



Amostra completa com classificação:


,nome,categoria,categoria_ia_simples
33,Extraordinário,livro,livro
3316,Fifa 2018 Narração Português Completo Midia D...,game,game
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo



Comparação (categoria original vs IA):


,nome,categoria,categoria_ia_simples,match_simples
33,Extraordinário,livro,livro,True
3316,Fifa 2018 Narração Português Completo Midia D...,game,game,True
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo,True
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem,True
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro,True
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo,True
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo,True
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game,True
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game,True
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo,False



Acurácia do modelo (prompt simples): 92.00%


### **2.7. Teste 2 - Prompt refinado**

No segundo teste, o prompt foi refinado com regras adicionais para orientar melhor a decisão do modelo em casos ambíguos.

Além da classificação, essa abordagem busca atuar como um mecanismo de validação, sendo mais conservadora quando o produto não se encaixa claramente em uma categoria ou quando o texto é insuficiente para uma decisão confiável.

In [18]:
# ============================================
# Teste 2 - Prompt refinado
# ============================================

prompt_refinado = ChatPromptTemplate.from_template("""
Você é um validador de cadastro de produtos para marketplace.

Sua tarefa é analisar o texto do produto (nome + descrição concatenados) e retornar apenas UMA categoria válida:
- o nome exato de uma categoria da lista; ou
- fora_escopo

Categorias válidas:
{categorias}

Definição das categorias:
- brinquedo: itens destinados à brincadeira ou recreação, incluindo brinquedos infantis, bonecos, kits lúdicos, fantasias e action figures, mesmo quando comercializados como itens colecionáveis
- game: produtos diretamente ligados a videogames, como consoles, controles, jogos eletrônicos, mídias físicas de videogame e acessórios relacionados a videogame
- livro: obras impressas ou publicações de leitura, como livros, coleções, boxes e materiais cuja função principal seja leitura
- maquiagem: produtos cosméticos ou estéticos, como batons, sombras, bases, kits de maquiagem e itens de embelezamento

Regras:
- Responda somente com o nome exato de uma categoria da lista ou com: fora_escopo
- Não explique a resposta
- Não crie categorias novas
- Analise apenas o campo "texto" que já contém nome + descrição do produto, portanto não há outras fontes
- Priorize a função principal do produto
- Ignore marca, cor, tamanho, volume, modelo, adjetivos promocionais e linguagem de venda

Critérios de decisão:
- Se o produto se encaixar claramente em uma categoria da lista, retorne essa categoria
- Se o produto não se encaixar claramente em nenhuma categoria, responda: fora_escopo
- Se faltar contexto para identificar o produto, responda: fora_escopo
- Se houver ambiguidade forte, responda: fora_escopo
- Se o texto for genérico, insuficiente ou inconsistente, responda: fora_escopo
- Se houver indício de item ilícito, proibido ou incompatível com um marketplace comum, responda: fora_escopo

Produto:
{texto}
""")

# Conecta o prompt ao modelo para execução
chain_refinado = prompt_refinado | llm

# Executa o prompt e valida a resposta do modelo
def classificar_produto_prompt_refinado(texto):
    response = chain_refinado.invoke({
        "categorias": ", ".join(categorias_validas),
        "texto": texto
    })
    return validar_resposta_categoria(response.content, categorias_validas)

# Teste: comportamento do modelo
df_teste_refinado = df_sample.head(10).copy()
df_teste_refinado["categoria_ia_refinado"] = df_teste_refinado["texto"].apply(classificar_produto_prompt_refinado)

print("Teste - comparação (10 casos):")
display(df_teste_refinado[["nome", "categoria", "categoria_ia_refinado"]])

# Aplicação na amostra completa
df_sample["categoria_ia_refinado"] = df_sample["texto"].apply(classificar_produto_prompt_refinado)

print("\nAmostra completa com classificação:")
display(df_sample[["nome", "categoria", "categoria_ia_refinado"]].head(10))

# Comparação da categoria original vs. IA
df_sample["match_refinado"] = df_sample["categoria"] == df_sample["categoria_ia_refinado"]

print("\nComparação (categoria original vs IA):")
display(df_sample[["nome", "categoria", "categoria_ia_refinado", "match_refinado"]].head(10))

# Acurácia: avaliação realizada na amostra de 100 registros
acuracia_refinado = df_sample["match_refinado"].mean()
print(f"\nAcurácia do modelo (prompt refinado): {acuracia_refinado:.2%}")

Teste - comparação (10 casos):


,nome,categoria,categoria_ia_refinado
33,Extraordinário,livro,livro
3316,Fifa 2018 Narração Português Completo Midia D...,game,game
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo



Amostra completa com classificação:


,nome,categoria,categoria_ia_refinado
33,Extraordinário,livro,livro
3316,Fifa 2018 Narração Português Completo Midia D...,game,game
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo



Comparação (categoria original vs IA):


,nome,categoria,categoria_ia_refinado,match_refinado
33,Extraordinário,livro,livro,True
3316,Fifa 2018 Narração Português Completo Midia D...,game,game,True
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo,True
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem,True
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro,True
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo,True
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo,True
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game,True
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game,True
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo,False



Acurácia do modelo (prompt refinado): 91.00%


### **2.8. Comparação dos resultados**
Compara o desempenho das duas abordagens lado a lado





In [19]:
# ============================================
# Comparação dos resultados
# ============================================

print("Acurácia - Prompt simples:", f"{acuracia_simples:.2%}")
print("Acurácia - Prompt refinado:", f"{acuracia_refinado:.2%}")

df_comparacao = df_sample[[
    "nome",
    "categoria",
    "categoria_ia_simples",
    "categoria_ia_refinado"
]].copy()

display(df_comparacao.head(10))

Acurácia - Prompt simples: 92.00%
Acurácia - Prompt refinado: 91.00%


,nome,categoria,categoria_ia_simples,categoria_ia_refinado
33,Extraordinário,livro,livro,livro
3316,Fifa 2018 Narração Português Completo Midia D...,game,game,game
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,brinquedo,brinquedo,brinquedo
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,maquiagem,maquiagem,maquiagem
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,livro,livro,livro
1444,Dragon Ball Z Majin Boo - Shodo Bandai,brinquedo,brinquedo,brinquedo
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,brinquedo,brinquedo,brinquedo
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,game,game,game
4005,Pes 2018 Original Ps3 100% Dublado Português ...,game,game,game
1736,Base- Stand -display Para Action Figure-pront...,brinquedo,fora_escopo,fora_escopo


### **2.9. Análise dos resultados**

O prompt simples apresentou acurácia superior (96%) em relação ao prompt refinado (93%), indicando maior aderência às categorias originais da base.

No entanto, essa comparação deve ser interpretada com cautela, pois a categoria do dataset foi atribuída por vendedores e pode conter inconsistências.

Nos testes, observou-se que o prompt simples tende a reproduzir com maior frequência o padrão do dataset, enquanto o prompt refinado adota um comportamento mais conservador, classificando como `fora_escopo` alguns casos em que o texto não oferece segurança suficiente para uma decisão.

Assim, a maior acurácia do prompt simples indica melhor aderência à base, mas não necessariamente maior capacidade de validação. Já o prompt refinado, embora menos aderente ao dataset, demonstra maior potencial para apoiar a identificação de possíveis inconsistências no cadastro.

### **2.10. Análise de divergências**

Nos casos em que os prompts divergiram, foi possível observar diferenças importantes de comportamento entre as abordagens.

O prompt simples tende a assumir uma categoria mesmo em cenários mais ambíguos, enquanto o prompt refinado é mais restritivo e prefere retornar `fora_escopo` quando não há evidência suficiente para uma classificação segura.

Esse comportamento é relevante para o problema do marketplace, pois nem sempre o melhor processo é o que mais replica a categoria existente, mas o que melhor ajuda a sinalizar cadastros que merecem revisão.

In [20]:
# ============================================
# Análise de divergência entre prompts
# ============================================

df_divergencia = df_sample[
    df_sample["categoria_ia_simples"] != df_sample["categoria_ia_refinado"]
].copy()

print(f"Quantidade de divergências entre os prompts: {len(df_divergencia)}")

display(df_divergencia[[
    "nome",
    "texto",
    "categoria",
    "categoria_ia_simples",
    "categoria_ia_refinado"
]].head(10))

Quantidade de divergências entre os prompts: 1


,nome,texto,categoria,categoria_ia_simples,categoria_ia_refinado
3318,"2.000 Gold Wow Gold Ouro Azralon (nemesis 2,0...","2.000 Gold Wow Gold Ouro Azralon (nemesis 2,0k...",game,game,fora_escopo


In [21]:
# ============================================
# Avaliação nos casos de divergência
# ============================================

df_divergencia["acerto_simples"] = (
    df_divergencia["categoria"] == df_divergencia["categoria_ia_simples"])

df_divergencia["acerto_refinado"] = (
    df_divergencia["categoria"] == df_divergencia["categoria_ia_refinado"])

print(f"Acurácia do simples nos casos de divergência: {df_divergencia['acerto_simples'].mean():.2%}")
print(f"Acurácia do refinado nos casos de divergência: {df_divergencia['acerto_refinado'].mean():.2%}")

Acurácia do simples nos casos de divergência: 100.00%
Acurácia do refinado nos casos de divergência: 0.00%


### **2.11. Conclusão da escolha do modelo**

Embora o prompt simples apresente maior acurácia, ele tende a reproduzir o padrão do dataset, incluindo possíveis inconsistências.

O prompt refinado, por outro lado, apresentou comportamento mais conservador, classificando como "fora_escopo" casos com maior ambiguidade ou baixa clareza.

Considerando o objetivo do problema de validar cadastros e mitigar inconsistências, a abordagem refinada se mostra mais adequada como mecanismo de apoio à revisão, mesmo com menor aderência ao dataset.

##**3. Processo final**


Nesta etapa, é definido o pipeline final de classificação com foco na validação de cadastros.

A abordagem escolhida prioriza a identificação de inconsistências, adotando um comportamento mais conservador para sinalizar produtos que não se encaixam claramente nas categorias definidas.

Neste trabalho, a coluna `validacao_categoria` indica concordância ou divergência em relação à categoria informada no dataset. Assim, os casos classificados como `divergente` devem ser interpretados como potenciais inconsistências para revisão manual, e não como erro absoluto da IA ou do vendedor.

###**3.1. Preparação do ambiente**

Para iniciar o pipeline, iremos preparar o ambiente instalando os pacotes e dependências necessárias além de definir as variáveis de acesso ao AzureOpenIA. Obs.: Necessário executar a sessão "API Key da OpenAI via Azure AI Foundry" do notebook para obter a API Key.

In [22]:
# ============================================
# Setup - Instalação de dependências (Opcional) e bibliotecas
# ============================================

print("Iniciando a preparação do ambiente...")

import pandas as pd
import numpy as np
import importlib

def instalar_se_nao_existir(pacote, nome_import=None):
    nome_import = nome_import or pacote
    if importlib.util.find_spec(nome_import) is None:
        print(f"Instalando {pacote}...")
        !pip install -q {pacote}

instalar_se_nao_existir("openai")
instalar_se_nao_existir("langchain")
instalar_se_nao_existir("langchain-openai", "langchain_openai")

Iniciando a preparação do ambiente...


In [23]:
# ============================================
# Setup - Definição das variáveis de ambiente Langchain | AzureOpenAI
# ============================================

# Obs.: Necessário executar a sessão "API Key da OpenAI via Azure AI Foundry" do notebook para obter a API Key

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_ENDPOINT,
    api_version=AZURE_API_VERSION,
    azure_deployment=AZURE_DEPLOYMENT_MODEL,
    temperature=0
    )

print("Ambiente configurado com sucesso!!")

Ambiente configurado com sucesso!!


###**3.2. Extração do dataset**

Realiza a extração do dataset com os dados de produtos diretamente do url da fonte e carrega em um DataFrame do Pandas.

In [24]:
# ============================================
# Extração - Base de Dados de Produtos (Completa)
# ============================================

dataset_path = "https://dados-ml-pln.s3-sa-east-1.amazonaws.com/produtos.csv"

# Load da base de produtos a partir da fonte externa
df = pd.read_csv(
    dataset_path,
    delimiter=";",
    encoding='utf-8')

print("Dataset extraído com sucesso!!")
print("Dimensão da base:")
print(f" - Total de Linhas: {df.shape[0]}")
print(f" - Total de Colunas: {df.shape[1]}")
print("\nVisualização dos primeiros 10 registros:")
display(df.head(10))

Dataset extraído com sucesso!!
Dimensão da base:
 - Total de Linhas: 4080
 - Total de Colunas: 3

Visualização dos primeiros 10 registros:


,nome,descricao,categoria
0,O Hobbit - 7ª Ed. 2013,Produto NovoBilbo Bolseiro é um hobbit que lev...,livro
1,Livro - It A Coisa - Stephen King,Produto NovoDurante as férias escolares de 195...,livro
2,Box As Crônicas De Gelo E Fogo Pocket 5 Li...,Produto NovoTodo o reino de Westeros ao alcanc...,livro
3,Box Harry Potter,Produto Novo e Físico A série Harry Potter ch...,livro
4,Livro Origem - Dan Brown,Produto NovoDe Onde Viemos? Para Onde Vamos? R...,livro
5,Mais Escuro - Cinquenta Tons Mais Escuros Pel...,Produto Novo e Físico O relacionamento quente...,livro
6,O Silmarillion - 5ª Ed. 2011,"Produto NovoO Silmarillion, relata acontecimen...",livro
7,O Pequeno Principe,O Pequeno Príncipe é um dos personagens mais f...,livro
8,Ed & Lorraine Warren - Demonologistas Arquiv...,Produto NovoEles enfrentaram os mistérios mais...,livro
9,Box - Franz Kafka 1883-1924 - 3 Livros,Produto NovoEste box contém 3 livros de Franz ...,livro


###**3.3. Processamento, limpeza e enriquecimento dos dados**
Executa todas as transformações e preparação dos dados para realizar as classificações dos produtos.

Realiza o tratamento para eliminar valores nulos do dataframe para transformações posteriores.

In [25]:
# ============================================
# Transformação - Tratamento de nulos
# ============================================

# Elimina os valores nulos de todas as colunas
df = df.fillna("")

# Validação
df_nulos = df.isnull().sum()

print("\nValores nulos:")
print(df_nulos)

if df_nulos.sum() > 0:
    raise ValueError("Ainda existem valores nulos no DataFrame!")


Valores nulos:
nome         0
descricao    0
categoria    0
dtype: int64


Adiciona uma nova coluna (texto) concatenando nome e descrição do produto, para fornecer mais contexto ao modelo de IA.

In [26]:
# ============================================
# Transformação - Criação da variável de entrada (texto)
# ============================================

df["texto"] = df["nome"] + " " + df["descricao"]
df["texto"] = df["texto"].str.strip()

display(df.head(10))

,nome,descricao,categoria,texto
0,O Hobbit - 7ª Ed. 2013,Produto NovoBilbo Bolseiro é um hobbit que lev...,livro,O Hobbit - 7ª Ed. 2013 Produto NovoBilbo Bols...
1,Livro - It A Coisa - Stephen King,Produto NovoDurante as férias escolares de 195...,livro,Livro - It A Coisa - Stephen King Produto Nov...
2,Box As Crônicas De Gelo E Fogo Pocket 5 Li...,Produto NovoTodo o reino de Westeros ao alcanc...,livro,Box As Crônicas De Gelo E Fogo Pocket 5 Liv...
3,Box Harry Potter,Produto Novo e Físico A série Harry Potter ch...,livro,Box Harry Potter Produto Novo e Físico A sér...
4,Livro Origem - Dan Brown,Produto NovoDe Onde Viemos? Para Onde Vamos? R...,livro,Livro Origem - Dan Brown Produto NovoDe Onde ...
5,Mais Escuro - Cinquenta Tons Mais Escuros Pel...,Produto Novo e Físico O relacionamento quente...,livro,Mais Escuro - Cinquenta Tons Mais Escuros Pelo...
6,O Silmarillion - 5ª Ed. 2011,"Produto NovoO Silmarillion, relata acontecimen...",livro,O Silmarillion - 5ª Ed. 2011 Produto NovoO Si...
7,O Pequeno Principe,O Pequeno Príncipe é um dos personagens mais f...,livro,O Pequeno Principe O Pequeno Príncipe é um do...
8,Ed & Lorraine Warren - Demonologistas Arquiv...,Produto NovoEles enfrentaram os mistérios mais...,livro,Ed & Lorraine Warren - Demonologistas Arquivo...
9,Box - Franz Kafka 1883-1924 - 3 Livros,Produto NovoEste box contém 3 livros de Franz ...,livro,Box - Franz Kafka 1883-1924 - 3 Livros Produt...


Gera uma amostra de 100 registros para facilitar a comparação dos resultados finais.

In [27]:
# ============================================
# Transformação - Criação da amostra (100 registros)
# ============================================

df_sample = df.sample(100, random_state=42)
display(df_sample.head(10))

,nome,descricao,categoria,texto
33,Extraordinário,Produto Novo“Extraordinário” é um livro que co...,livro,Extraordinário Produto Novo“Extraordinário” é...
3316,Fifa 2018 Narração Português Completo Midia D...,FIFA 2018 - XBOX 360 MIDIA DIGITAL COMPLETOATE...,game,Fifa 2018 Narração Português Completo Midia Di...
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,Descrição do Kit Ninja (Foto):Uma Kunai Asuma ...,brinquedo,Kit Mega Sarutobi Kunai Naruto Asuma Shurikens...
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,CAIXA C/ 50 CORES DIFERENTES!BATOM MATTE QUEEN...,maquiagem,Caixa 50 Cores Batom Queen Fosco Matte Nude Ki...
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,A série Crônicas de Gelo e apresentações. São ...,livro,Box Dourado Crónica De Gelo E Fogo Edição De C...
1444,Dragon Ball Z Majin Boo - Shodo Bandai,- Dragon Ball Z Shodo- Figura Majin Boo- Fabri...,brinquedo,Dragon Ball Z Majin Boo - Shodo Bandai - Drag...
1029,Bonecos Vingadores Marvel Avengers Dc Luz E S...,O ENVIO MAIS RÁPIDO DO MERCADO LIVRE!DE 1 A 12...,brinquedo,Bonecos Vingadores Marvel Avengers Dc Luz E So...
3880,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo ...,POR FAVOR NÃO ESQUEÇA DE SELECIONAR A OPÇÃO RE...,game,Fifa 18 Fifa 2018 Ps3 Psn Mídia Digital Jogo D...
4005,Pes 2018 Original Ps3 100% Dublado Português ...,DescriçãoAqui Nascem Lendas' apresenta a volta...,game,Pes 2018 Original Ps3 100% Dublado Português B...
1736,Base- Stand -display Para Action Figure-pront...,Base -stand-Display para action figure -figura...,brinquedo,Base- Stand -display Para Action Figure-pronta...


###**3.4. Classificação dos produtos**
Utiliza a AzureOpenIA para criar uma análise de classificação dos produtos e comparar com os dados inseridos pelos anunciantes.

Obtém as categorias existentes na base para serem usados de referência pela IA.

In [28]:
# ============================================
# Preparação - Categorias válidas
# ============================================

categorias_validas = sorted(df_sample["categoria"].dropna().unique().tolist())
print(categorias_validas)

['brinquedo', 'game', 'livro', 'maquiagem']


Cria funções auxiliares para refinar as respostas da IA.

In [29]:
# ============================================
# Preparação - Funções auxiliares
# ============================================

def normalizar_categoria(texto):
    # Padroniza a saída do modelo (lowercase, remoção de caracteres) para evitar erros na comparação
    if texto is None:
        return ""
    return (
        str(texto)
        .strip()
        .lower()
        .replace(".", "")
        .replace("`", "")
        .replace('"', "")
    )

def validar_resposta_categoria(resposta, categorias_validas):
    # Verifica se a resposta do modelo está entre as categorias válidas.
    # Caso contrário, retorna "fora_escopo"
    resposta = normalizar_categoria(resposta)
    if resposta in categorias_validas:
        return resposta
    return "fora_escopo"

Define o prompt de classificação e a função para aplicação no dataframe.

In [30]:
# ============================================
# Classificação - Definicação do prompt e função
# ============================================

# Função final baseada na abordagem escolhida que foca em validação (função "classificar_produto_prompt_refinado" conforme a sessão 2.7)

prompt_refinado = ChatPromptTemplate.from_template("""
Você é um validador de cadastro de produtos para marketplace.

Sua tarefa é analisar o texto do produto (nome + descrição concatenados) e retornar apenas UMA categoria válida:
- o nome exato de uma categoria da lista; ou
- fora_escopo

Categorias válidas:
{categorias}

Definição das categorias:
- brinquedo: itens destinados à brincadeira ou recreação, incluindo brinquedos infantis, bonecos, kits lúdicos, fantasias e action figures, mesmo quando comercializados como itens colecionáveis
- game: produtos diretamente ligados a videogames, como consoles, controles, jogos eletrônicos, mídias físicas de videogame e acessórios relacionados a videogame
- livro: obras impressas ou publicações de leitura, como livros, coleções, boxes e materiais cuja função principal seja leitura
- maquiagem: produtos cosméticos ou estéticos, como batons, sombras, bases, kits de maquiagem e itens de embelezamento

Regras:
- Responda somente com o nome exato de uma categoria da lista ou com: fora_escopo
- Não explique a resposta
- Não crie categorias novas
- Analise apenas o campo "texto" que já contém nome + descrição do produto, portanto não há outras fontes
- Priorize a função principal do produto
- Ignore marca, cor, tamanho, volume, modelo, adjetivos promocionais e linguagem de venda

Critérios de decisão:
- Se o produto se encaixar claramente em uma categoria da lista, retorne essa categoria
- Se o produto não se encaixar claramente em nenhuma categoria, responda: fora_escopo
- Se faltar contexto para identificar o produto, responda: fora_escopo
- Se houver ambiguidade forte, responda: fora_escopo
- Se o texto for genérico, insuficiente ou inconsistente, responda: fora_escopo
- Se houver indício de item ilícito, proibido ou incompatível com um marketplace comum, responda: fora_escopo

Produto:
{texto}
""")

# Conecta o prompt ao modelo para execução
chain_refinado = prompt_refinado | llm

# Executa o prompt e valida a resposta do modelo
def classificar_produto_prompt_refinado(texto):
    response = chain_refinado.invoke({
        "categorias": ", ".join(categorias_validas),
        "texto": texto
    })
    return validar_resposta_categoria(response.content, categorias_validas)

Aplica a função no sample e gera o relatório final com as classificações da IA e comparação com as classificações dos anunciantes.

In [31]:
# ============================================
# Aplicação da função final apenas no sample de 100 registros
# ============================================

df_resultado_final = df_sample.copy()

df_resultado_final["categoria_genai"] = df_resultado_final["texto"].apply(classificar_produto_prompt_refinado)

df_resultado_final["validacao_categoria"] = df_resultado_final.apply(
    lambda row: "correta" if row["categoria_genai"] == row["categoria"] else "divergente",
    axis=1)

display(
    df_resultado_final[
        ["nome", "descricao", "categoria", "texto", "categoria_genai", "validacao_categoria"]
    ])

,nome,descricao,categoria,texto,categoria_genai,validacao_categoria
33,Extraordinário,Produto Novo“Extraordinário” é um livro que co...,livro,Extraordinário Produto Novo“Extraordinário” é...,livro,correta
3316,Fifa 2018 Narração Português Completo Midia D...,FIFA 2018 - XBOX 360 MIDIA DIGITAL COMPLETOATE...,game,Fifa 2018 Narração Português Completo Midia Di...,game,correta
1557,Kit Mega Sarutobi Kunai Naruto Asuma Shuriken...,Descrição do Kit Ninja (Foto):Uma Kunai Asuma ...,brinquedo,Kit Mega Sarutobi Kunai Naruto Asuma Shurikens...,brinquedo,correta
2548,Caixa 50 Cores Batom Queen Fosco Matte Nude K...,CAIXA C/ 50 CORES DIFERENTES!BATOM MATTE QUEEN...,maquiagem,Caixa 50 Cores Batom Queen Fosco Matte Nude Ki...,maquiagem,correta
457,Box Dourado Crónica De Gelo E Fogo Edição De ...,A série Crônicas de Gelo e apresentações. São ...,livro,Box Dourado Crónica De Gelo E Fogo Edição De C...,livro,correta
...,...,...,...,...,...,...
3615,Grand Theft Auto V Gta 5 Em Portugues Midia F...,GTA 5: confira a análise da versão HD do game ...,game,Grand Theft Auto V Gta 5 Em Portugues Midia Fi...,game,correta
3882,Ufc 3 Ps4 Psn Original 1 Mídia Digital Portug...,,game,Ufc 3 Ps4 Psn Original 1 Mídia Digital Português,game,correta
251,Evangelho De Sangue,Produto NovoTodo mal tem uma origem. Pinhead e...,livro,Evangelho De Sangue Produto NovoTodo mal tem ...,livro,correta
270,"Lote C/ 50 Livros Kit Diversos P/ Sebo, Bibli...","""LOTE COM 50 LIVROS USADOS""""Temas variados""Lit...",livro,"Lote C/ 50 Livros Kit Diversos P/ Sebo, Biblio...",livro,correta


###**3.5. Carregamento do relatório final**
Gera o arquivo CSV com os dados do relatório final (Sample com 100 registros).

In [32]:
# ============================================
# Exportação do arquivo final em CSV
# ============================================

colunas_exportacao = [
    "nome",
    "descricao",
    "categoria",
    "texto",
    "categoria_genai",
    "validacao_categoria"
]

df_resultado_final[colunas_exportacao].to_csv(
    "data_refined/produtos_classificados_genai.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig")

print("Arquivo 'produtos_classificados_genai.csv' gerado com sucesso.")

Arquivo 'produtos_classificados_genai.csv' gerado com sucesso.


### **3.5. Resumo da validação final**

A etapa final resume a quantidade de casos em que a categoria prevista pela IA coincide ou diverge da categoria originalmente informada, permitindo identificar o volume de registros potencialmente inconsistentes para revisão manual.

In [33]:
# ============================================
# Resumo final da validação
# ============================================

resumo_validacao = (
    df_resultado_final["validacao_categoria"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="quantidade"))

resumo_validacao["percentual"] = (
    resumo_validacao["quantidade"] / len(df_resultado_final) * 100
).round(2)

display(resumo_validacao)

,status,quantidade,percentual
0,correta,91,91.0
1,divergente,9,9.0


## **4. Conclusão**

Este trabalho demonstrou que a IA Generativa pode ser utilizada não apenas para classificar produtos, mas como um mecanismo de validação automática de cadastros em um marketplace.

O pipeline desenvolvido compara a categoria sugerida pela IA com a categoria informada pelo vendedor, permitindo identificar possíveis inconsistências de forma automatizada.

Embora o prompt simples apresente maior aderência ao dataset, ele tende a reproduzir possíveis erros existentes na base. Já o prompt refinado adota um comportamento mais conservador, sendo mais adequado para sinalizar produtos que não se encaixam claramente nas categorias definidas.

Dessa forma, a solução proposta funciona como um "fiscal automático", capaz de:

- identificar divergências entre a classificação do vendedor e a da IA  
- sinalizar produtos que podem estar categorizados incorretamente  
- reduzir o esforço manual na revisão de cadastros  

> Assim, o valor do processo não está apenas na classificação, mas na sua capacidade de apoiar a validação e melhorar a qualidade dos dados no marketplace.

### **4.1. Possíveis extensões do processo**

Além da validação de categorias, o mesmo processo pode ser estendido para identificar outros problemas comuns em marketplaces, como:

- produtos com descrições insuficientes ou inconsistentes  
- itens potencialmente ilícitos ou não permitidos  
- produtos com informações enganosas ou irrelevantes  

Essas extensões poderiam ser implementadas com regras adicionais no prompt ou com múltiplos modelos especializados, ampliando o papel da IA como mecanismo de governança de dados.

No entanto, neste trabalho, o foco foi mantido na validação de categorias, conforme o escopo proposto.